# COMP9414 Assignment 1 — Search and Planning
**zID:** z5555555 (replace with your zID)

**Due:** Friday 3 July 2026, 5:00 PM AEST

## Setup and Imports

In [1]:
import sys
!{sys.executable} -m pip install "ai9414>=0.1.4"

  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached uvicorn-0.49.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 16.3 MB/s eta 0:00:00
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.1 MB/s eta 0:00:00a 0:00:01
Using cached uvicorn-0.49.0-py3-none-any.whl (71 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB

In [2]:
from __future__ import annotations

import json
import math
import heapq
from collections import deque
from pathlib import Path
from typing import Any

# Planning helpers (requires: pip install ai9414>=0.1.4)
from ai9414.strips import (
    apply_action_signature,
    get_applicable_actions,
    get_initial_facts,
)

## Helper: Load Data Files

In [4]:
DATA_DIR = Path("data")

def load_graph(name: str) -> dict:
    """
    This is assuming that data for assignments live under ${pwd}/data folder.
    """
    with open(DATA_DIR / "graphs" / name) as f:
        return json.load(f)

def load_planning(name: str) -> dict:
    """Load a planning JSON file from data/planning/."""
    with open(DATA_DIR / "planning" / name) as f:
        return json.load(f)

# Quick test
office = load_graph("office.json")
print(f"Office graph: {len(office['nodes'])} nodes, {len(office['edges'])} edges")
print(f"Start: {office['start']}, Goal: {office['goal']}")

Office graph: 6 nodes, 8 edges
Start: corridor, Goal: lab


---
# Part A: Problem Formulation and Hand Search Traces

## 4.1 Office Graph Formulation

In [5]:
from IPython.display import Image
Image(url="https://i.imgur.com/OeRA9qR.png")

### Q1: State the initial state, goal test, actions, transition model, and path cost for this graph.

- Initial state: `corridor`
- Goal test: `state == lab`
- Actions: Move along any edge to an adjacent room (bidirectional)
- Transition model: Moving from room A to room B changes the current state to B
- Path cost: Sum of edge costs along the path

### Q2: Draw or tabulate the first three frontier updates for DFS, BFS, and UCS.

#### DFS

DFS (stack, LIFO. push in reverse alphabetical so alphabetically first is popped)

| Step | Pop       | Frontier                                                                                       |
|------|-----------|------------------------------------------------------------------------------------------------|
| 0    | -         | [corridor]                                                                                     |
| 1    | corridor  | [mail_room, office_a, office_b]                                                                |
| 2    | mail_room | [storage, office_a, office_b] <- mail_room's unvisited neighbours [visited(corridor), storage] |
| 3    | storage   | [lab, office_a, office_b] <- storage's unvisited neighbours [lab, visited(mail_room)]          |
| 4    | lab       | GOAL_FOUND                                                                                     |


#### BFS

queue, FIFO — push alphabetically, pop from front

| Step | Dequeue   | Visited                                   | Queue                           |
|------|-----------|-------------------------------------------|---------------------------------|
| 0    |           | {corridor}                                | [corridor]                      |
| 1    | corridor  | {corridor, mail_room, office_a, office_b} | [mail_room, office_a, office_b] |
| 2    | mail_room | + { storage }                             | [office_a, office_b, storage]   |
| 3    | office_a  | + { lab }                                 | [office_b, storage, lab]        |
| 4    | office_b  | (no new unvisited)                        | [storage, lab]                  |
| 5    | storage   | (lab already visited)                     | [lab]                           |
| 6    | lab       | GOAL FOUND                                |                                 |



#### UCS

min-heap by cumulative cost

| Step | Pop       | Cost | Heap After                                                                                                                                                                                |
|------|-----------|------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| 0    |           |      | [(0, corridor)]                                                                                                                                                                           |
| 1    | corridor  | 0    | [(2, mail_room), (2, office_a), (3, office_b)]                                                                                                                                            |
| 2    | mail_room | 2    | [(2, office_a), (3, office_b), (4, storage)]                                                                                                                                              |
| 3    | office_a  | 2    | [(3, office_b), (4, storage), (4, office_b via office_a), (10, lab)] - but office_b: new cost 2+2=4 > 3 existing, so no relax. lab: 2+8=10, push [(3, office_b), (4, storage), (10, lab)] |
| 4    | office_b  | 3    | [(4, storage), (6, lab), (10, lab)] ← lab via office_b: 3+3=6 < 10 RELAX                                                                                                                  |
| 5    | storage   | 4    | [(6, lab), (10, lab), (11, lab)] ← lab via storage: 4+7=11 > 6 NO RELAX. 11 > 6 so don't push [(6, lab), (10, lab)]                                                                       |
| 6    | lab       | 6    | GOAL FOUND                                                                                                                                                                                |


### Q3: Report the full expansion order for DFS, BFS, and UCS.

#### DFS

DFS expansion order: corridor, mail_room, storage, lab

#### BFS

BFS expansion order: corridor, mail_room, office_a, office_b, storage, lab

#### UCS

UCS expansion order: corridor, mail_room, office_a, office_b, storage, lab


### Q4: Report the returned path and returned cost for each algorithm.


#### DFS

DFS path: corridor → mail_room → storage → lab

DFS cost: 2 + 2 + 7 = 11

#### BFS


BFS path: corridor → office_a → lab (parent of lab = office_a, parent of office_a = corridor)

BFS cost: 2 + 8 = 10

#### UCS

UCS path: corridor → office_b → lab

UCS cost: 3 + 3 = 6

### Q5: Explain why BFS and UCS can return different paths on this graph.

BFS finds the shallowest path (fewest edges) corridor -> office_a -> lab has only 2 edges - that's the minimum but it costs 2 + 8 = 10
UCS finds the cheapest path (lowest cost) corridor -> office_b -> lab has 2 edges but costs 3 + 3 = 6. UCS discovers this because it expands office_b (costs 3) before office_a's edge (costs 10) gets popped

BFS doesn't care about costs, it picks office_a because it's enqueued before office_b alpahbatically. UCS picks office_b -> lab because 6 < 10

Key insights: BFS is optimal only when all edge costs are same. When costs vary, UCS wins.

## 4.2 Search tree versus state space

State space graph: each room appears exactly once as a node. The office graph has 6 nodes (corridor, mail_room, office_a, office_b, storage, lab) and 8 edges.

Search tree: each node represents a state + the path to reach it. The same room can appear multiple times because there are multiple paths to reach it

---
# Part B: Implement Graph Search

## 5.1 `solve_graph(problem, algorithm)`

In [55]:
def build_adjacency(graph: dict[str, Any]) -> dict[str, list[str]]:
    """
    Build an adjacency list from the graph dictionary.

    The graph uses this format:
        {
            "nodes": [{"id": "A", "x": 0.1, "y": 0.2}, ...],
            "edges": [{"u": "A", "v": "B"}, ...],
            "start": "A",
            "goal": "G"
        }
    """
    adjacency = {str(node["id"]): [] for node in graph["nodes"]}
    for edge in graph["edges"]:
        left = str(edge["u"])
        right = str(edge["v"])
        adjacency[left].append(right)
        adjacency[right].append(left)
    for node_id in adjacency:
        adjacency[node_id].sort()
    return adjacency


def build_adjacency_with_weight(graph: dict[str, Any]) -> dict[str, list[tuple[str, float]]]:
    """
    Build an adjacency list from the weighted graph dictionary.

    The graph uses this format:
        {
            "nodes": [{"id": "A", "x": 0.1, "y": 0.2}, ...],
            "edges": [{"u": "A", "v": "B", "cost": 1.7}, ...],
            "start": "A",
            "goal": "G"
        }
    """
    adjacency = {str(node["id"]): [] for node in graph["nodes"]}
    for edge in graph["edges"]:
        left = str(edge["u"])
        right = str(edge["v"])
        cost = float(edge["cost"])
        adjacency[left].append((right, cost))
        adjacency[right].append((left, cost))
    for node_id in adjacency:
        adjacency[node_id].sort(key=lambda item: (item[1], item[0]))
    return adjacency


def heuristic(graph: dict, node: str, goal: str) -> float:
    """Euclidean distance from node to goal (straight-line heuristic for A*)."""
    positions = {str(n["id"]): (float(n["x"]), float(n["y"])) for n in graph["nodes"]}
    return math.dist(positions[node], positions[goal])


def get_neighbours(adjacency: dict[str, list[tuple[str, float]]], node_id: str) -> list[tuple[str, float]]:
    """
    Return neighbouring nodes in deterministic UCS order.

    Important:
        The order matters. The demo expects a fixed order for a fixed graph.
    """
    return list(adjacency[node_id])


def normalise_node_id(raw: Any) -> str:
    """
    Convert one node id into a Python string.

    Example:
        normalise_node_id("A") returns "A".
    """
    return str(raw)



def reconstruct_path(parents: dict[str, str | None], goal: str) -> list[str]:
    """
    Reconstruct the final path from the start node to the goal node.
    """
    path: list[str] = []
    current: str | None = goal
    while current is not None:
        path.append(current)
        current = parents[current]
    path.reverse()
    return path



def solve_graph(problem: dict, algorithm: str) -> dict:
    """
    Solve a graph search problem using the specified algorithm.
    
    Args:
        problem: Graph dict with nodes, edges, start, goal.
        algorithm: One of 'dfs', 'bfs', 'ucs', 'astar'.
    
    Returns:
        Dict with: status, path, plan, cost, expanded_order,
                   expanded_count, generated_count, frontier_peak.
    """
    start = normalise_node_id(problem["start"])
    goal = normalise_node_id(problem["goal"])
    adjacency = build_adjacency(problem)
    adjacency_with_weights = build_adjacency_with_weight(problem)
    frontier_peak = 1
    
    # --- DFS ---
    if algorithm == "dfs":
        stack = [start]
        visited = {start}
        parents = {start: None}
        visited_order = [start]

        while stack:
            current = stack[-1]  # peek at the top — this is our current node

            # Goal check: if the top of the stack is the goal, we're done
            if current == goal:
                return {
                    "algorithm": "dfs",
                    "status": "found",
                    "path": reconstruct_path(parents, goal),
                    "visited_order": visited_order,
                }

            # Find the first unvisited neighbour (sorted order for determinism)
            neighbors = get_neighbours(adjacency, current)
            next_node = None
            for n in neighbors:
                if n not in visited:
                    next_node = n
                    break  # DFS only takes the FIRST unvisited neighbour
    
            if next_node is not None:
                # EXPAND: push the neighbour and go deeper
                stack.append(next_node)
                visited.add(next_node)
                parents[next_node] = current
                visited_order.append(next_node)
    
            else:
                # BACKTRACK: no unvisited neighbours — dead end, pop and retreat
                stack.pop()
    
    # --- BFS ---
    elif algorithm == "bfs":
        queue = deque([start])  # FIFO frontier — pop from left, push to right
        visited = {start}  # nodes we've already seen (never revisit)
        parents = {start: None}  # parent pointers for path reconstruction
        depths = {start: 0}  # track each node's depth for the trace
        visited_order = [start]  # discovery order for the trace

        while queue:
            node = queue.popleft()  # take the OLDEST node (FIFO = breadth-first)
    
            # Goal check: if the node we just dequeued is the goal, we're done
            if node == goal:
                path = reconstruct_path(parents, goal)
                return {
                    "algorithm": "bfs",
                    "status": "found",
                    "path": path,
                    "visited_order": visited_order,
                }
    
            # Expand: enqueue ALL unvisited neighbours (sorted for determinism)
            neighbors = get_neighbours(adjacency, node)
            for neighbor in neighbors:
                if neighbor not in visited:
                    visited.add(neighbor)
                    parents[neighbor] = node
                    depths[neighbor] = depths[node] + 1
                    queue.append(neighbor)
                    visited_order.append(neighbor)
                    route = reconstruct_path(parents, neighbor)
        
    # --- UCS ---
    elif algorithm == "ucs":

        frontier = [(0, start)]
        parents: dict[str, str | None] = {start: None}  # for path reconstruction
        visited = set()  # nodes whose cheapest cost is finalised (popped from heap)
        best_costs: dict[str, float] = {start: 0.0}  # cheapest known cost to reach each node
        generated_count = 1
        visited_order: list[str] = []
    
        while frontier:
            frontier_peak = max(frontier_peak, len(frontier))  
            # Pop the cheapest node from the frontier (min-heap property)
            cost, node = heapq.heappop(frontier)
    
            # Skip stale entries: if we already finalised this node via a cheaper path, this heap entry is outdated — ignore it
            if node in visited:
                continue

            # Finalise this node: once popped, its cost is guaranteed optimal
            visited.add(node)
            visited_order.append(node)
           
            if node == goal:
                return {
                    "status": "found",
                    "path": reconstruct_path(parents, goal),
                    "plan": [],
                    "cost": cost,
                    "expanded_order": visited_order,
                    "expanded_count": len(visited_order),
                    "generated_count": generated_count,
                    "frontier_peak": frontier_peak,
                }

            for neighbor, edge_cost in get_neighbours(adjacency_with_weights, node):
                new_cost = cost + edge_cost
                # Only push if this is a NEW node or we found a CHEAPER path
                if neighbor not in best_costs or new_cost < best_costs[neighbor]:
                    best_costs[neighbor] = new_cost
                    parents[neighbor] = node
                    # Push the new (cost, node) pair — old entries become stale
                    heapq.heappush(frontier, (new_cost, neighbor))
            
      
        
        return {
            "status": "not_found", "path": [], "plan": [], "cost": None,
            "expanded_order": expanded_order, "expanded_count": len(expanded_order),
            "generated_count": generated_count, "frontier_peak": frontier_peak,
        }
    
    # --- A* ---
    elif algorithm == "astar":
        counter = 0
        h_start = heuristic(problem, start, goal)
        frontier = [(h_start, [start], counter, start)]
        best_g: dict[str, float] = {start: 0.0}
        visited = set()
        expanded_order = []
        generated_count = 1
        frontier_peak = 1
        
        while frontier:
            frontier_peak = max(frontier_peak, len(frontier))
            f_val, path, _, node = heapq.heappop(frontier)
            
            if node in visited:
                continue
            visited.add(node)
            expanded_order.append(node)
            
            g_val = best_g[node]
            
            if node == goal:
                return {
                    "status": "found",
                    "path": path,
                    "plan": [],
                    "cost": g_val,
                    "expanded_order": expanded_order,
                    "expanded_count": len(expanded_order),
                    "generated_count": generated_count,
                    "frontier_peak": frontier_peak,
                }
            
            for neighbour, edge_cost in adjacency_with_weights[node]:
                new_g = g_val + edge_cost
                if new_g < best_g.get(neighbour, float("inf")):
                    best_g[neighbour] = new_g
                    counter += 1
                    new_path = path + [neighbour]
                    new_f = new_g + heuristic(problem, neighbour, goal)
                    heapq.heappush(frontier, (new_f, new_path, counter, neighbour))
                    generated_count += 1
        
        return {
            "status": "not_found", "path": [], "plan": [], "cost": None,
            "expanded_order": expanded_order, "expanded_count": len(expanded_order),
            "generated_count": generated_count, "frontier_peak": frontier_peak,
        }
    
    else:
        return {
            "status": "error",
            "path": [], "plan": [], "cost": None,
            "expanded_order": [], "expanded_count": 0,
            "generated_count": 0, "frontier_peak": 0,
            "message": f"Unknown algorithm: {algorithm}",
        }

### Quick Test: solve_graph on office

In [57]:
office = load_graph("office.json")
for alg in ["dfs", "bfs", "ucs", "astar"]:
    result = solve_graph(office, alg)
    # print(f"{alg:6s} | status={result['status']} | cost={result['cost']} | path={result['path']}")
    # print(f"       | expanded={result['expanded_order']} | expanded_count={result['expanded_count']}")
    # print(f"       | generated_count={result['generated_count']} | frontier_peak={result['frontier_peak']}")
    print(result)

{'algorithm': 'dfs', 'status': 'found', 'path': ['corridor', 'mail_room', 'storage', 'lab'], 'visited_order': ['corridor', 'mail_room', 'storage', 'lab']}
{'algorithm': 'bfs', 'status': 'found', 'path': ['corridor', 'office_a', 'lab'], 'visited_order': ['corridor', 'mail_room', 'office_a', 'office_b', 'storage', 'lab']}
{'status': 'found', 'path': ['corridor', 'office_b', 'lab'], 'plan': [], 'cost': 6.0, 'expanded_order': ['corridor', 'mail_room', 'office_a', 'office_b', 'storage', 'lab'], 'expanded_count': 6, 'generated_count': 1, 'frontier_peak': 3}
{'status': 'found', 'path': ['corridor', 'office_b', 'lab'], 'plan': [], 'cost': 6.0, 'expanded_order': ['corridor', 'office_b', 'office_a', 'mail_room', 'lab'], 'expanded_count': 5, 'generated_count': 6, 'frontier_peak': 3}


## 5.2 Evaluation: Run All Algorithms on All Graphs

In [ ]:
graph_files = ["office.json", "graph2.json", "graph3.json"]
algorithms = ["dfs", "bfs", "ucs", "astar"]

all_results = {}

print(f"{'Graph':<14} {'Alg':<6} {'Status':<10} {'Cost':<8} {'Expanded':<10} {'Generated':<11} {'Peak':<6} Path")
print("-" * 100)

for gf in graph_files:
    graph = load_graph(gf)
    for alg in algorithms:
        r = solve_graph(graph, alg)
        all_results[(gf, alg)] = r
        cost_str = f"{r['cost']:.1f}" if r['cost'] is not None else "None"
        path_str = " → ".join(r['path']) if r['path'] else "—"
        print(f"{gf:<14} {alg:<6} {r['status']:<10} {cost_str:<8} {r['expanded_count']:<10} {r['generated_count']:<11} {r['frontier_peak']:<6} {path_str}")

### Comparison Plot: Expanded Count by Algorithm

In [ ]:
# Simple text-based bar chart (no matplotlib required)
# Replace with matplotlib if you prefer a visual plot.

for gf in graph_files:
    print(f"\n--- {gf} ---")
    for alg in algorithms:
        r = all_results[(gf, alg)]
        bar = "█" * r["expanded_count"]
        print(f"  {alg:<6} expanded={r['expanded_count']:>3}  cost={r['cost']}  {bar}")

---
# Part C: A* Heuristic Behaviour (1.5 marks)

## 6.1 Straight-line Heuristic

The heuristic is Euclidean distance: $h(n) = \sqrt{(x_n - x_g)^2 + (y_n - y_g)^2}$

This is **admissible** because straight-line distance never exceeds the actual path cost (triangle inequality).

## 6.2 Empirical Comparison: UCS vs A*

In [ ]:
# Compare UCS vs A* on graph1 (small) and graph2 (larger)
comparison_graphs = ["graph1.json", "graph2.json"]

print(f"{'Graph':<14} {'Alg':<6} {'Cost':<8} {'Expanded':<10} {'Generated':<11} {'Peak':<6}")
print("-" * 60)

for gf in comparison_graphs:
    graph = load_graph(gf)
    for alg in ["ucs", "astar"]:
        r = solve_graph(graph, alg)
        cost_str = f"{r['cost']:.3f}" if r['cost'] is not None else "None"
        print(f"{gf:<14} {alg:<6} {cost_str:<8} {r['expanded_count']:<10} {r['generated_count']:<11} {r['frontier_peak']:<6}")
    print()

### Analysis

TODO: 
1. Identify a point in the trace where A* chooses a different frontier entry from UCS because of h(n).
2. Explain how the heuristic changed the search effort relative to UCS.
3. Discuss one limitation of Euclidean distance when edge costs ≠ geometric distance.

---
# Part D: STRIPS Modelling and Forward Planning (3.0 marks)

## 7.1 Office Delivery Domain

Objects: `robot`, `parcel`, `keycard`, `lab_door`  
Rooms: `corridor`, `mail_room`, `office_a`, `office_b`, `lab`  
Goal: `at(parcel, lab)`

Actions: `move`, `pickup_keycard`, `pickup_parcel`, `unlock_door`, `drop_parcel`

## 7.2 Plan Diagnosis: `lab_via_office_b`

Proposed plan:
1. `move(robot, corridor, mail_room)`
2. `pickup_parcel(robot, parcel, mail_room)`
3. `move(robot, mail_room, corridor)`
4. `move(robot, corridor, office_b)`
5. `move(robot, office_b, lab)` ← **FAILS**
6. `drop_parcel(robot, parcel, lab)`

### Q1: First action that is not applicable

TODO

### Q2: Missing precondition or false fact

TODO

### Q3: Why inserting unlock_door alone is not enough

TODO

### Q4: Valid repaired plan

TODO

### Q5: Where unlocked(lab_door) first becomes true

TODO

### Q6: How the final state satisfies at(parcel, lab)

TODO

## 7.3 Forward BFS Planner: `solve_planning(problem)`

In [ ]:
def solve_planning(problem: dict) -> dict:
    """
    Forward state-space BFS planner using STRIPS.
    
    Args:
        problem: Planning problem dict.
    
    Returns:
        Dict with: status, path, plan, cost, expanded_order,
                   expanded_count, generated_count, frontier_peak.
    """
    def state_id(facts):
        """Stable canonical id for a state (sorted tuple of fact tuples)."""
        return tuple(sorted(facts))
    
    def goal_satisfied(facts, goal):
        """True when every goal fact is present in the current state."""
        return set(tuple(g) for g in goal).issubset(set(tuple(f) for f in facts))
    
    start = get_initial_facts(problem)
    goal = problem["goal"]
    
    frontier = deque([(start, [])])  # (facts, plan)
    visited = {state_id(start)}
    expanded_order = []
    generated_count = 1
    frontier_peak = 1
    
    while frontier:
        frontier_peak = max(frontier_peak, len(frontier))
        facts, plan = frontier.popleft()
        
        sid = state_id(facts)
        expanded_order.append(sid)
        
        if goal_satisfied(facts, goal):
            return {
                "status": "found",
                "path": [],
                "plan": plan,
                "cost": len(plan),
                "expanded_order": expanded_order,
                "expanded_count": len(expanded_order),
                "generated_count": generated_count,
                "frontier_peak": frontier_peak,
            }
        
        for action in get_applicable_actions(problem, facts):
            next_facts = apply_action_signature(problem, facts, action)
            next_sid = state_id(next_facts)
            if next_sid not in visited:
                visited.add(next_sid)
                frontier.append((next_facts, plan + [action]))
                generated_count += 1
    
    return {
        "status": "not_found",
        "path": [], "plan": [], "cost": None,
        "expanded_order": expanded_order, "expanded_count": len(expanded_order),
        "generated_count": generated_count, "frontier_peak": frontier_peak,
    }

### Test: solve_planning on canonical_delivery

In [ ]:
cd = load_planning("canonical_delivery.json")
result_cd = solve_planning(cd)
print(f"Status: {result_cd['status']}")
print(f"Plan ({result_cd['cost']} actions):")
for i, action in enumerate(result_cd['plan'], 1):
    print(f"  {i}. {action}")
print(f"Expanded: {result_cd['expanded_count']}, Generated: {result_cd['generated_count']}, Frontier peak: {result_cd['frontier_peak']}")

## 7.4 Planning Comparison

In [ ]:
planning_files = ["canonical_delivery.json", "unlocked_lab.json"]

print(f"{'Problem':<25} {'Status':<10} {'Plan Len':<10} {'Expanded':<10} {'Generated':<11} {'Peak':<6}")
print("-" * 75)

planning_results = {}
for pf in planning_files:
    p = load_planning(pf)
    r = solve_planning(p)
    planning_results[pf] = r
    cost_str = str(r['cost']) if r['cost'] is not None else "None"
    print(f"{pf:<25} {r['status']:<10} {cost_str:<10} {r['expanded_count']:<10} {r['generated_count']:<11} {r['frontier_peak']:<6}")

print("\n--- Plans ---")
for pf in planning_files:
    r = planning_results[pf]
    print(f"\n{pf}:")
    for i, action in enumerate(r['plan'], 1):
        print(f"  {i}. {action}")

### Why the unlocked case needs fewer actions

TODO: Explain why unlocked_lab can be solved with fewer actions than canonical_delivery.

---
# 9.1 Reusable Evaluation Cell

This cell loads all data files, runs all algorithms, prints a results table, and writes `zID_results.json`.

In [ ]:
# ==================== REUSABLE EVALUATION CELL ====================
# This cell must be self-contained. It calls solve_graph and solve_planning
# defined above, runs them on all supplied data files, and writes results.

import json
from pathlib import Path

DATA_DIR = Path("data")
ZID = "z5555555"  # <-- REPLACE WITH YOUR zID

# --- Graph search ---
graph_files = {
    "office": "graphs/office.json",
    "graph1": "graphs/graph1.json",
    "graph2": "graphs/graph2.json",
    "graph3": "graphs/graph3.json",
}
algorithms = ["dfs", "bfs", "ucs", "astar"]

results = {"graph_search": {}, "planning": {}}

print("=" * 100)
print("GRAPH SEARCH RESULTS")
print("=" * 100)
print(f"{'Graph':<10} {'Alg':<6} {'Status':<10} {'Cost':<10} {'Expanded':<10} {'Generated':<11} {'Peak':<6} Path")
print("-" * 100)

for name, filepath in graph_files.items():
    with open(DATA_DIR / filepath) as f:
        graph = json.load(f)
    results["graph_search"][name] = {}
    for alg in algorithms:
        r = solve_graph(graph, alg)
        results["graph_search"][name][alg] = r
        cost_str = f"{r['cost']:.3f}" if r['cost'] is not None else "None"
        path_str = " → ".join(r['path'][:6])
        if len(r['path']) > 6:
            path_str += " → ..."
        print(f"{name:<10} {alg:<6} {r['status']:<10} {cost_str:<10} {r['expanded_count']:<10} {r['generated_count']:<11} {r['frontier_peak']:<6} {path_str}")

# --- Planning ---
planning_files = {
    "canonical_delivery": "planning/canonical_delivery.json",
    "unlocked_lab": "planning/unlocked_lab.json",
}

print("\n" + "=" * 100)
print("PLANNING RESULTS")
print("=" * 100)
print(f"{'Problem':<25} {'Status':<10} {'Plan Len':<10} {'Expanded':<10} {'Generated':<11} {'Peak':<6}")
print("-" * 75)

for name, filepath in planning_files.items():
    with open(DATA_DIR / filepath) as f:
        problem = json.load(f)
    r = solve_planning(problem)
    # Convert expanded_order tuples to strings for JSON serialization
    r_json = dict(r)
    r_json["expanded_order"] = [str(s) for s in r["expanded_order"]]
    results["planning"][name] = r_json
    cost_str = str(r['cost']) if r['cost'] is not None else "None"
    print(f"{name:<25} {r['status']:<10} {cost_str:<10} {r['expanded_count']:<10} {r['generated_count']:<11} {r['frontier_peak']:<6}")
    print(f"  Plan: {r['plan']}")

# --- Write results JSON ---
output_path = f"{ZID}_results.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"\nResults written to {output_path}")